# Grad-CAM Analysis

This notebook loads the three dropout models from W&B artifacts, runs Grad-CAM on a small fixed set of test images, shows the results locally, and logs them back to W&B.


## 1. Setup and Environment Check

This cell installs W&B for the active kernel, imports the libraries we need, and resolves the project root.

In [1]:
%pip install wandb


In [2]:
import os
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import wandb

IS_COLAB = "COLAB_GPU" in os.environ

if IS_COLAB:
    PROJECT_ROOT = Path("/content/DAT255_Bayesian")
else:
    PROJECT_ROOT = Path.cwd().resolve()
    if not (PROJECT_ROOT / "requirements.txt").exists() and (PROJECT_ROOT.parent / "requirements.txt").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")


Project root: /content/DAT255_Bayesian
TensorFlow version: 2.19.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Project Configuration

This cell defines the same lightweight configuration and artifact names used by the training notebook.

In [ ]:
@dataclass(frozen=True)
class ProjectConfig:
    """Collect the main settings used throughout the notebook."""
    image_size: tuple[int, int] = (128, 128)
    batch_size: int = 32
    num_classes: int = 10
    mc_samples: int = 30
    dropout_rate: float = 0.3
    conv_filters: tuple[int, ...] = (32, 64, 128)
    dense_units: int = 128
    raw_data_dir: Path = field(default_factory=lambda: PROJECT_ROOT / "data" / "raw")


def get_config() -> ProjectConfig:
    """Return one shared config object for the full experiment."""
    return ProjectConfig()


config = get_config()

WANDB_PROJECT = "dat255-bayesian"
WANDB_ENTITY = None
WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "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")
GRADCAM_RUN_NAME = "notebook-05-gradcam-analysis"

MODEL_ARTIFACTS = [
    {"artifact_name": "notebook-04-dropout-0-1-model", "dropout_rate": 0.1},
    {"artifact_name": "notebook-04-dropout-0-3-model", "dropout_rate": 0.3},
    {"artifact_name": "notebook-04-dropout-0-5-model", "dropout_rate": 0.5},
]

# Set to 1 for a quick smoke test. Use 3 for the normal class-balanced analysis view.
GRADCAM_SAMPLES_PER_CLASS = 3

if WANDB_API_KEY == "PASTE_YOUR_API_KEY_HERE":
    raise ValueError("Set WANDB_API_KEY before running the notebook.")

wandb.login(key=WANDB_API_KEY)
print(config)


## 3. Data Loading Utilities

This cell reuses the same ImageNette loading logic as notebook 4 so the Grad-CAM analysis uses the same test split.

In [5]:
IMAGENETTE_URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"
IMAGENETTE_MD5 = "e793b78cc4c9e9a4ccc0c1155377a412"
IMAGENETTE_FOLDER_NAME = "imagenette2-160"
DATASET_SEED = 255


def find_imagenette_dir(raw_data_dir: Path) -> Path | None:
    """Find the extracted ImageNette folder inside the raw data directory."""
    candidates = [
        raw_data_dir / IMAGENETTE_FOLDER_NAME,
        raw_data_dir / f"{IMAGENETTE_FOLDER_NAME}.tgz" / IMAGENETTE_FOLDER_NAME,
    ]

    for candidate in candidates:
        if (candidate / "train").is_dir() and (candidate / "val").is_dir():
            return candidate

    return None


def ensure_imagenette_dataset(raw_data_dir: Path) -> Path:
    """Reuse ImageNette if it already exists, otherwise download it once."""
    raw_data_dir.mkdir(parents=True, exist_ok=True)
    dataset_dir = find_imagenette_dir(raw_data_dir)

    if dataset_dir is not None:
        print(f"ImageNette already exists at: {dataset_dir}")
        return dataset_dir

    print("ImageNette not found. Downloading to data/raw...")
    tf.keras.utils.get_file(
        origin=IMAGENETTE_URL,
        file_hash=IMAGENETTE_MD5,
        cache_dir=str(raw_data_dir),
        cache_subdir="",
        extract=True,
    )

    dataset_dir = find_imagenette_dir(raw_data_dir)
    if dataset_dir is None:
        raise FileNotFoundError("ImageNette was downloaded, but the dataset folder could not be found.")

    print(f"ImageNette downloaded to: {dataset_dir}")
    return dataset_dir


def load_datasets(config: ProjectConfig) -> tuple[tf.data.Dataset, tf.data.Dataset, tf.data.Dataset]:
    """Create train, validation, and test datasets for the experiment."""
    imagenette_dir = ensure_imagenette_dataset(config.raw_data_dir)

    train_dir = imagenette_dir / "train"
    test_dir = imagenette_dir / "val"

    train_ds = tf.keras.utils.image_dataset_from_directory(
        directory=str(train_dir),
        labels="inferred",
        label_mode="categorical",
        batch_size=config.batch_size,
        image_size=config.image_size,
        validation_split=0.2,
        subset="training",
        shuffle=True,
        seed=DATASET_SEED,
    ).prefetch(tf.data.AUTOTUNE)

    val_ds = tf.keras.utils.image_dataset_from_directory(
        directory=str(train_dir),
        labels="inferred",
        label_mode="categorical",
        batch_size=config.batch_size,
        image_size=config.image_size,
        validation_split=0.2,
        subset="validation",
        shuffle=True,
        seed=DATASET_SEED,
    ).prefetch(tf.data.AUTOTUNE)

    test_ds = tf.keras.utils.image_dataset_from_directory(
        directory=str(test_dir),
        labels="inferred",
        label_mode="categorical",
        batch_size=config.batch_size,
        image_size=config.image_size,
        shuffle=False,
        seed=DATASET_SEED,
    ).prefetch(tf.data.AUTOTUNE)

    print("Datasets loaded successfully.")
    return train_ds, val_ds, test_ds


## 4. Model and Uncertainty Helpers

This cell reuses the same custom dropout layer and MC prediction helper so the artifact models can be loaded correctly.

In [6]:
class MCDropout(tf.keras.layers.Dropout):
    """Dropout layer that stays active during inference for MC sampling."""

    def call(self, inputs, training=None):
        return super().call(inputs, training=True)


def mc_predict(
    model: tf.keras.Model,
    images: tf.Tensor,
    mc_samples: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Run several stochastic forward passes and summarize the predictions."""
    predictions = []
    for _ in range(mc_samples):
        predictions.append(model(images, training=False).numpy())

    stacked = np.stack(predictions, axis=0)
    mean_prediction = stacked.mean(axis=0)
    variance = stacked.var(axis=0)
    predictive_entropy = -np.sum(
        mean_prediction * np.log(mean_prediction + 1e-10),
        axis=1,
    )
    return mean_prediction, variance, predictive_entropy


## 5. W&B Artifact Loading

This cell downloads the saved models from W&B artifacts and prepares a fixed set of test images that all three models will share.

In [ ]:
def artifact_reference(artifact_name: str, alias: str = "latest") -> str:
    api = wandb.Api()
    entity = WANDB_ENTITY or getattr(api, "default_entity", None)
    if entity:
        return f"{entity}/{WANDB_PROJECT}/{artifact_name}:{alias}"
    return f"{WANDB_PROJECT}/{artifact_name}:{alias}"


def download_artifact_file(artifact_name: str, alias: str = "latest") -> Path:
    artifact = wandb.Api().artifact(artifact_reference(artifact_name, alias))
    download_root = PROJECT_ROOT / "artifacts" / "wandb" / "downloads" / artifact_name
    artifact_dir = Path(artifact.download(root=str(download_root)))
    files = [item for item in artifact_dir.iterdir() if item.is_file()]
    if len(files) != 1:
        raise FileNotFoundError(
            f"Expected exactly one file in artifact {artifact_name}, found {len(files)}."
        )
    return files[0]


def load_model_from_artifact(artifact_name: str, alias: str = "latest") -> tf.keras.Model:
    model_path = download_artifact_file(artifact_name, alias)
    print(f"Loaded model artifact: {artifact_name}:{alias} -> {model_path}")
    return tf.keras.models.load_model(
        model_path,
        custom_objects={"MCDropout": MCDropout},
    )


def select_images_per_class(
    dataset: tf.data.Dataset,
    samples_per_class: int,
    class_names: list[str],
) -> tuple[tf.Tensor, tf.Tensor, list[int]]:
    """Collect a fixed number of test images from each class in dataset order."""
    class_counts = {index: 0 for index in range(len(class_names))}
    selected_images = []
    selected_labels = []
    selected_class_order = []

    for batch_images, batch_labels in dataset:
        true_indices = batch_labels.numpy().argmax(axis=1)

        for image, label, true_index in zip(batch_images, batch_labels, true_indices):
            if class_counts[int(true_index)] >= samples_per_class:
                continue

            selected_images.append(image.numpy())
            selected_labels.append(label.numpy())
            selected_class_order.append(int(true_index))
            class_counts[int(true_index)] += 1

        if all(count >= samples_per_class for count in class_counts.values()):
            break

    sample_images = tf.convert_to_tensor(np.stack(selected_images), dtype=tf.float32)
    sample_labels = tf.convert_to_tensor(np.stack(selected_labels), dtype=tf.float32)
    return sample_images, sample_labels, selected_class_order


imagenette_dir = ensure_imagenette_dataset(config.raw_data_dir)
class_names = sorted(path.name for path in (imagenette_dir / "train").iterdir() if path.is_dir())

train_ds, val_ds, test_ds = load_datasets(config)

sample_images, sample_labels, selected_class_order = select_images_per_class(
    test_ds,
    samples_per_class=GRADCAM_SAMPLES_PER_CLASS,
    class_names=class_names,
)
true_class_indices = sample_labels.numpy().argmax(axis=1)

print(
    f"Using {len(sample_images)} fixed test images: "
    f"{GRADCAM_SAMPLES_PER_CLASS} per class across {len(class_names)} classes."
)


## 6. Grad-CAM Helpers

These helpers keep the Grad-CAM implementation short and readable: find the last convolution layer, build a heatmap, and overlay it on the original image.

In [8]:
def find_last_conv_layer_name(model: tf.keras.Model) -> str:
    """Return the name of the last Conv2D layer in the model."""
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    raise ValueError("No Conv2D layer was found in the model.")


def make_gradcam_heatmap(
    model: tf.keras.Model,
    image_tensor: tf.Tensor,
    class_index: int,
    conv_layer_name: str,
) -> np.ndarray:
    """Create a Grad-CAM heatmap for one image and one target class."""
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(conv_layer_name).output, model.output],
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(image_tensor, training=False)
        target_score = predictions[:, class_index]

    gradients = tape.gradient(target_score, conv_outputs)
    pooled_gradients = tf.reduce_mean(gradients, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_gradients, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    max_value = tf.reduce_max(heatmap)
    if float(max_value) > 0:
        heatmap = heatmap / max_value
    return heatmap.numpy()


def overlay_gradcam_on_image(image: np.ndarray, heatmap: np.ndarray, alpha: float = 0.4) -> np.ndarray:
    """Overlay a Grad-CAM heatmap on top of the original RGB image."""
    image = image.astype(np.float32) / 255.0
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], image.shape[:2]).numpy().squeeze()
    colored_heatmap = plt.colormaps["jet"](heatmap_resized)[..., :3]
    overlay = np.clip((1 - alpha) * image + alpha * colored_heatmap, 0, 1)
    return (overlay * 255).astype(np.uint8)


## 7. Run Grad-CAM for the Three Artifact Models

This cell loads each model, runs prediction and MC-dropout uncertainty on the same fixed images, builds Grad-CAM overlays, and stores the results for local plots and W&B logging.

In [9]:
gradcam_results = []

for artifact_config in MODEL_ARTIFACTS:
    artifact_name = artifact_config["artifact_name"]
    dropout_rate = artifact_config["dropout_rate"]

    model = load_model_from_artifact(artifact_name)
    conv_layer_name = find_last_conv_layer_name(model)

    mean_prediction, variance, predictive_entropy = mc_predict(
        model,
        sample_images,
        config.mc_samples,
    )

    predicted_class_indices = mean_prediction.argmax(axis=1)
    confidence_scores = mean_prediction.max(axis=1)
    mean_variances = variance.mean(axis=1)

    for sample_index in range(len(sample_images)):
        image = sample_images[sample_index].numpy().astype(np.uint8)
        target_class_index = int(predicted_class_indices[sample_index])
        heatmap = make_gradcam_heatmap(
            model,
            tf.expand_dims(sample_images[sample_index], axis=0),
            class_index=target_class_index,
            conv_layer_name=conv_layer_name,
        )
        overlay = overlay_gradcam_on_image(image, heatmap)

        gradcam_results.append(
            {
                "artifact_name": artifact_name,
                "dropout_rate": dropout_rate,
                "sample_index": sample_index,
                "true_class": class_names[int(true_class_indices[sample_index])],
                "predicted_class": class_names[target_class_index],
                "confidence": float(confidence_scores[sample_index]),
                "predictive_entropy": float(predictive_entropy[sample_index]),
                "mean_variance": float(mean_variances[sample_index]),
                "raw_image": image,
                "gradcam_overlay": overlay,
            }
        )

print(f"Collected {len(gradcam_results)} Grad-CAM result rows.")


wandb:   1 of 1 files downloaded.  


Loaded model artifact: notebook-04-dropout-0-1-model:latest -> /content/DAT255_Bayesian/artifacts/wandb/downloads/notebook-04-dropout-0-1-model/dropout_0_1.keras


wandb:   1 of 1 files downloaded.  


Loaded model artifact: notebook-04-dropout-0-3-model:latest -> /content/DAT255_Bayesian/artifacts/wandb/downloads/notebook-04-dropout-0-3-model/dropout_0_3.keras


wandb:   1 of 1 files downloaded.  


Loaded model artifact: notebook-04-dropout-0-5-model:latest -> /content/DAT255_Bayesian/artifacts/wandb/downloads/notebook-04-dropout-0-5-model/dropout_0_5.keras
Collected 18 Grad-CAM result rows.


## 8. Visualize and Log to W&B

This final cell shows the Grad-CAM overlays side by side in the notebook and uploads both a table and one combined figure to W&B.

In [ ]:
gradcam_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=GRADCAM_RUN_NAME,
    job_type="analysis",
    config={
        "samples_per_class": GRADCAM_SAMPLES_PER_CLASS,
        "total_samples": int(len(sample_images)),
    },
    reinit="finish_previous",
)

gradcam_table = wandb.Table(columns=[
    "artifact_name",
    "dropout_rate",
    "sample_index",
    "true_class",
    "predicted_class",
    "confidence",
    "predictive_entropy",
    "mean_variance",
    "raw_image",
    "gradcam_overlay",
])

for row in gradcam_results:
    gradcam_table.add_data(
        row["artifact_name"],
        row["dropout_rate"],
        row["sample_index"],
        row["true_class"],
        row["predicted_class"],
        row["confidence"],
        row["predictive_entropy"],
        row["mean_variance"],
        wandb.Image(row["raw_image"]),
        wandb.Image(row["gradcam_overlay"]),
    )

num_rows = len(sample_images)
num_cols = len(MODEL_ARTIFACTS)
fig, axes = plt.subplots(num_rows, num_cols, figsize=(5 * num_cols, 3.2 * num_rows))
axes = np.atleast_2d(axes)

for row_index in range(num_rows):
    for col_index, artifact_config in enumerate(MODEL_ARTIFACTS):
        ax = axes[row_index, col_index]
        result = next(
            item for item in gradcam_results
            if item["sample_index"] == row_index and item["artifact_name"] == artifact_config["artifact_name"]
        )
        ax.imshow(result["gradcam_overlay"])
        ax.axis("off")
        ax.set_title(
            f"d={result['dropout_rate']} | true={result['true_class']}\n"
            f"pred={result['predicted_class']} | conf={result['confidence']:.2f}\n"
            f"H={result['predictive_entropy']:.3f} | var={result['mean_variance']:.5f}",
            fontsize=9,
        )

plt.tight_layout()
plt.show()

wandb.log({
    "gradcam_comparison_table": gradcam_table,
    "gradcam_comparison_figure": wandb.Image(fig),
})

wandb.finish()
